# 05 Silver — `fact_urna_event`

Transforma a camada Bronze `fact_urna_logs` (mensagens de texto dos logs de urna) na tabela Silver `fact_urna_event`, removendo carryover do 1º turno em partições do 2º turno (com proteção contra *clock drift*), classificando semanticamente cada linha de log contra o catálogo declarativo de eventos (`event_catalog.yml`) e descartando todo texto livre e PII.

### Objetivos:
- Ler todos os Parquet Bronze particionados em `election_year/turn/uf` sem alterar a Bronze 1:1.
- Remover linhas de carryover do 1º turno presentes em arquivos `.jez` do 2º turno via marcadores de ciclo de vida (`file_seq`), histórico de `urna_serial` do 1º turno e fallback de data apenas em timestamps não-drifted.
- Preservar 100% dos eventos do 1º turno e todos os eventos legítimos do 2º turno com *clock drift* de BIOS (`is_clock_drift = True`).
- Classificar cada linha com `EventCatalog.classify()` (allowlist semântica, zero-PII).
- Produzir Silver Parquet com 17 colunas tipadas (incluindo `is_clock_drift`, `event_code`, `classification_status`, `biometric_attempt_num`, `parser_version`).
- Reconciliar conservação de linhas (`bronze_rows == carryover_rows_dropped + silver_rows_kept`) e emitir auditoria por `(uf, turn)` no manifesto.

In [ ]:
import importlib
import importlib.util
import inspect
import os
import shutil
import stat
import subprocess
import sys
from pathlib import Path

_REQUIRED_PACKAGES = (
    ("kagglehub", "kagglehub>=0.4.0"),
    ("polars", "polars"),
    ("pyarrow", "pyarrow"),
    ("duckdb", "duckdb"),
    ("yaml", "pyyaml"),
)
if globals().get("__name__") == "__main__" and not os.getenv("PYTEST_CURRENT_TEST"):
    for _mod_name, _pip_name in _REQUIRED_PACKAGES:
        if importlib.util.find_spec(_mod_name) is None:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "-U", _pip_name]
            )
            importlib.invalidate_caches()
    import kagglehub as _kh_check

    if "output_dir" not in inspect.signature(_kh_check.dataset_download).parameters:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-U", "kagglehub>=0.4.0"]
        )
        importlib.invalidate_caches()
        sys.modules.pop("kagglehub", None)
        import kagglehub as _kh_check

# Contrato único de estado e ciclo de vida via KaggleHub SDK (Kaggle e Google Colab)
DATASET_HANDLE = os.getenv("KAGGLE_DATASET_HANDLE", "daniellmattos/tse-eleicoes-dados-v2")
DATASET_PATH = os.getenv("DATASET_PATH", "/content/drive/MyDrive/tse-eleicoes-dados/data")
STAGING_PATH = os.getenv("STAGING_PATH", "/tmp/tse-eleicoes-staging")
ALLOW_COLD_START = False


def is_directory_empty(path: Path) -> bool:
    """Verifica se DATASET_PATH está efetivamente vazio (ignorando arquivos ocultos de sistema)."""
    if not path.exists():
        return True
    return not any(item for item in path.iterdir() if not item.name.startswith("."))


def ensure_writable_tree(root: Path) -> None:
    """Garante permissão de escrita em árvores copiadas de mounts somente-leitura."""
    if not root.exists():
        return
    root.chmod(stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
    for dirpath, dirnames, filenames in os.walk(root):
        base = Path(dirpath)
        for dname in dirnames:
            dp = base / dname
            if not dp.is_symlink():
                dp.chmod(stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
        for fname in filenames:
            fp = base / fname
            if not fp.is_symlink():
                fp.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IRGRP | stat.S_IROTH)


def is_dataset_not_found_error(exc: BaseException) -> bool:
    """Detecta erros de dataset inexistente (404 / Not Found) para permitir reconstrução do zero."""
    msg = f"{type(exc).__name__}: {exc}".lower()
    not_found_markers = (
        "404",
        "not found",
        "does not exist",
        "no dataset found",
        "resource not found",
        "403",
    )
    return any(marker in msg for marker in not_found_markers)


def ensure_kaggle_auth() -> None:
    """Autentica via KaggleHub SDK sem travar execuções headless (batch kernels / CI)."""
    import kagglehub

    has_env_creds = bool(
        os.getenv("KAGGLE_API_TOKEN")
        or (os.getenv("KAGGLE_USERNAME") and os.getenv("KAGGLE_KEY"))
        or (Path.home() / ".kaggle" / "kaggle.json").exists()
        or Path("/root/.config/kaggle/kaggle.json").exists()
    )
    if has_env_creds:
        print("[AUTH] Credenciais do Kaggle detectadas no ambiente.")
        return
    if sys.stdin is not None and sys.stdin.isatty():
        print("Iniciando autenticação manual no Kaggle SDK...")
        kagglehub.login()
    else:
        print(
            "[AUTH] Execução não-interativa detectada; pulando prompt interativo kagglehub.login()."
        )


def bootstrap_dataset(
    handle: str, target_dir: Path, allow_cold_start: bool = False
) -> Path:
    """Se DATASET_PATH estiver vazio, baixa a última versão do dataset do Kaggle ou reconstrói do zero se não existir."""
    target_dir.mkdir(parents=True, exist_ok=True)
    if not is_directory_empty(target_dir):
        print(f"[OK] DATASET_PATH ({target_dir}) já contém dados locais. Pulando download.")
        return target_dir

    print(
        f"[BOOTSTRAP] DATASET_PATH ({target_dir}) está vazio. Baixando '{handle}' via Kaggle SDK..."
    )
    try:
        import kagglehub

        try:
            downloaded_path = Path(
                kagglehub.dataset_download(
                    handle,
                    output_dir=str(target_dir),
                    force_download=True,
                )
            )
        except TypeError as type_err:
            if "output_dir" not in str(type_err):
                raise
            downloaded_path = Path(
                kagglehub.dataset_download(
                    handle,
                    force_download=True,
                )
            )

        if downloaded_path.resolve() != target_dir.resolve() and downloaded_path.exists():
            print(
                f"[SYNC] Copiando arquivos de {downloaded_path} (read-only mount) para {target_dir}..."
            )
            shutil.copytree(downloaded_path, target_dir, dirs_exist_ok=True)
            ensure_writable_tree(target_dir)

        print(f"[OK] Dataset populado com sucesso em {target_dir}.")
    except Exception as exc:
        if allow_cold_start or is_dataset_not_found_error(exc):
            print(
                f"[COLD-START] Dataset '{handle}' não encontrado ou ainda não publicado ({exc}). "
                f"Iniciando reconstrução do zero em {target_dir}."
            )
        else:
            raise RuntimeError(
                f"Falha ao popular DATASET_PATH ({target_dir}) a partir do Kaggle ('{handle}'). "
                "Verifique sua autenticação e conexão de rede."
            ) from exc

    return target_dir


if globals().get("__name__") == "__main__" and not os.getenv("PYTEST_CURRENT_TEST"):
    _drive_mount_point = Path("/content/drive")
    if not (_drive_mount_point / "MyDrive").exists():
        try:
            from google.colab import drive

            drive.mount(str(_drive_mount_point))
        except Exception as _exc:
            print(f"[BOOT] Headless/CLI execution detected while checking Drive ({_exc}).")

    _token_cache_file = Path("/root/.config/kaggle/colab_token.env")
    if not os.environ.get("KAGGLE_API_TOKEN"):
        if _token_cache_file.exists():
            os.environ["KAGGLE_API_TOKEN"] = _token_cache_file.read_text(encoding="utf-8").strip()
        else:
            try:
                from google.colab import userdata

                _token = userdata.get("KAGGLE_API_TOKEN")
                if _token:
                    os.environ["KAGGLE_API_TOKEN"] = _token.strip()
                    _token_cache_file.parent.mkdir(parents=True, exist_ok=True)
                    _token_cache_file.write_text(_token.strip(), encoding="utf-8")
                    _token_cache_file.chmod(0o600)
            except Exception as _exc:
                print(
                    f"[BOOT] userdata unavailable in CLI mode ({_exc}); "
                    "relying on environment variables."
                )
    Path(DATASET_PATH).mkdir(parents=True, exist_ok=True)
    Path(STAGING_PATH).mkdir(parents=True, exist_ok=True)
    ensure_kaggle_auth()
    bootstrap_dataset(
        DATASET_HANDLE, Path(DATASET_PATH), allow_cold_start=ALLOW_COLD_START
    )

import json
import re

import polars as pl
import yaml

ELECTION_YEAR = 2026
TURNS = (
    [int(s.strip()) for s in os.environ["TSE_TARGET_TURNS"].split(",") if s.strip()]
    if os.environ.get("TSE_TARGET_TURNS")
    else [1, 2]
)
ALL_UFS = [
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA",
    "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN",
    "RO", "RR", "RS", "SC", "SE", "SP", "TO", "ZZ",
]
TARGET_UFS = (
    [s.strip().upper() for s in os.environ["TSE_TARGET_UFS"].split(",") if s.strip()]
    if os.environ.get("TSE_TARGET_UFS")
    else list(ALL_UFS)
)
PARSER_VERSION = "1.1.0"


def find_table_dir(table_name, roots=(Path(DATASET_PATH),), max_depth=7):
    """Localiza <table_name>/ com partições election_year=* sob DATASET_PATH."""
    for root in roots:
        if not root.exists():
            continue
        base = len(root.parts)
        for dirpath, dirnames, _ in os.walk(root):
            p = Path(dirpath)
            if len(p.parts) - base >= max_depth:
                dirnames[:] = []
                continue
            if p.name == table_name and any(d.startswith("election_year=") for d in dirnames):
                return p
    return None


BRONZE_BASE = find_table_dir("fact_urna_logs")
if BRONZE_BASE is None:
    raise FileNotFoundError(
        f"Tabela Bronze fact_urna_logs não encontrada em {DATASET_PATH}. "
        f"Execute a Fase 03 ou verifique o dataset {DATASET_HANDLE}."
    )
print(f"BRONZE_BASE resolvido: {BRONZE_BASE}")
SILVER_OUT = Path(DATASET_PATH) / "silver" / "fact_urna_event"
SILVER_OUT.mkdir(parents=True, exist_ok=True)

print(f"Setup concluído. Polars {pl.__version__}, {len(TARGET_UFS)}/{len(ALL_UFS)} UFs, turnos={TURNS}, parser={PARSER_VERSION}.")

In [ ]:
# ---------------------------------------------------------------------------
# Catálogo declarativo de eventos — embutido inline (zero-PII, allowlist)
# ---------------------------------------------------------------------------

EVENT_CATALOG_YAML = r"""
schema_version: "1.0"
fallback_event_code: "UNKNOWN_EVENT"
malformed_event_code: "MALFORMED_LINE"
allowed_extractors:
  - "biometric_attempt_num"

required_event_codes:
  - "LOGD_STARTED"
  - "SECTION_OPENED"
  - "VOTER_ENABLED"
  - "BIOMETRIC_READ_ATTEMPT"
  - "BIOMETRIC_RECOGNIZED"
  - "BIOMETRIC_NOT_RECOGNIZED"
  - "NO_BIOMETRIC_REGISTERED"
  - "MANUAL_ENABLE_PATH"
  - "INACTIVITY_TIMEOUT_45S"
  - "INVALID_KEY_PRESSED"
  - "CARGO_VOTE_RECORDED"
  - "VOTER_VOTE_COMPUTED"
  - "VOTING_SUSPENDED"
  - "VOTING_RESUMED"
  - "CONTINGENCY_EVENT"
  - "SECTION_CLOSED"

patterns:
  - pattern_id: "logd_started_main"
    event_code: "LOGD_STARTED"
    applications: ["*"]
    regex: "(?i)(?:in[ií]cio\\s+das\\s+opera[cç][oõ]es\\s+do\\s+logd|logd\\s+iniciado)"
    extractors: {}
    description: "Inicialização do serviço de log (logd) da urna eletrônica."

  - pattern_id: "section_opened_main"
    event_code: "SECTION_OPENED"
    applications: ["VOTA", "GAP", "*"]
    regex: "(?i)(?:urna\\s+pronta\\s+para\\s+receber\\s+votos|in[ií]cio\\s+da\\s+vota[cç][aã]o|vota[cç][aã]o\\s+iniciada|abertura\\s+da\\s+se[cç][aã]o|emiss[aã]o\\s+d[ae]\\s+zer[eé]sima)"
    extractors: {}
    description: "Abertura da seção eleitoral, emissão da Zerésima ou início da votação."

  - pattern_id: "voter_enabled_main"
    event_code: "VOTER_ENABLED"
    applications: ["VOTA", "*"]
    regex: "(?i)^(?=.*\\beleitor\\s+(?:foi\\s+)?habilitado\\b)(?!.*\\b(?:sem\\s+biometria|manual)\\b).*$"
    extractors: {}
    description: "Habilitação do eleitor para iniciar a sessão de votação."

  - pattern_id: "biometric_read_attempt_numbered"
    event_code: "BIOMETRIC_READ_ATTEMPT"
    applications: ["VOTA", "*"]
    regex: "(?i)^(?!(?:.*\\b(?:mes[aá]rio|presidente|autoteste|teste\\s+de\\s+sensor)\\b))(?:.*(?:tentativa\\s+de\\s+(?:reconhecimento|leitura|identifica[cç][aã]o)\\s+biom[eé]tric[oa]|solicita(?:ndo)?\\s+digital[.:;]?\\s*tentativa|capturada\\s+a\\s+digital[.:;]?\\s*tentativa|n[uú]mero\\s+de\\s+tentativas\\s+de\\s+reconhecimento\\s+do\\s+dedo(?:[.:;]?\\s*tentativa)?)\\s*(?:n[uúº°.]*\\s*)?[:=\\-]?\\s*[\\[\\(]?(?P<biometric_attempt_num>\\d{1,3})[\\]\\)]?(?:\\b|$))"
    extractors:
      biometric_attempt_num: "biometric_attempt_num"
    description: "Tentativa numerada de leitura ou reconhecimento biométrico da digital do eleitor (incluindo formatos oficiais TSE 'Solicita digital. Tentativa [N]', 'Capturada a digital. Tentativa [N]' e 'Número de tentativas de reconhecimento do dedo')."

  - pattern_id: "biometric_read_attempt_generic"
    event_code: "BIOMETRIC_READ_ATTEMPT"
    applications: ["VOTA", "*"]
    regex: "(?i)^(?!(?:.*\\b(?:mes[aá]rio|presidente|autoteste|teste\\s+de\\s+sensor)\\b))(?:.*(?:tentativa\\s+de\\s+(?:reconhecimento|leitura|identifica[cç][aã]o)\\s+biom[eé]tric[oa]|solicita(?:[cç][aã]o\\s+de)?\\s+(?:digital|biometria)(?:\\s+do\\s+eleitor)?|aguardando\\s+(?:leitura\\s+biom[eé]trica|digital\\s+do\\s+eleitor)|n[uú]mero\\s+de\\s+tentativas\\s+de\\s+reconhecimento\\s+do\\s+dedo))"
    extractors: {}
    description: "Solicitação ou tentativa de leitura biométrica do eleitor sem número explícito na mensagem."

  - pattern_id: "biometric_not_recognized_main"
    event_code: "BIOMETRIC_NOT_RECOGNIZED"
    applications: ["VOTA", "*"]
    regex: "(?i)^(?!(?:.*\\b(?:mes[aá]rio|presidente|autoteste|teste\\s+de\\s+sensor)\\b))(?:.*(?:(?:digital|biometria|dedo)(?:\\s+do\\s+eleitor|\\s+capturad[oa])?\\s+n[aã]o\\s+(?:foi\\s+)?reconhecid[oa]|timeout\\s+de\\s+reconhecimento\\s+do\\s+dedo|falha\\s+no\\s+reconhecimento\\s+biom[eé]trico|impress[aã]o\\s+digital\\s+n[aã]o\\s+confere))"
    extractors: {}
    description: "Falha, timeout ou não reconhecimento da impressão digital do eleitor na tentativa biométrica."

  - pattern_id: "biometric_recognized_main"
    event_code: "BIOMETRIC_RECOGNIZED"
    applications: ["VOTA", "*"]
    regex: "(?i)^(?!(?:.*\\b(?:mes[aá]rio|presidente|autoteste|teste\\s+de\\s+sensor)\\b))(?:.*(?:(?<!n[aã]o\\s)(?<!n[aã]o\\sfoi\\s)\\b(?:biometria|digital|dedo)(?:\\s+do\\s+eleitor)?\\s+reconhecid[oa]\\b|tipo\\s+de\\s+habilita[cç][aã]o\\s+do\\s+eleitor\\s*\\[?biom[eé]trica\\]?|reconhecimento\\s+biom[eé]trico\\s+(?:efetuado\\s+com\\s+sucesso|positivo|ok)|identifica[cç][aã]o\\s+biom[eé]trica\\s+positiva|eleitor\\s+identificado\\s+por\\s+biometria))"
    extractors: {}
    description: "Reconhecimento biométrico bem-sucedido da impressão digital do eleitor (incluindo 'Tipo de habilitação do eleitor [biométrica]')."

  - pattern_id: "no_biometric_registered_main"
    event_code: "NO_BIOMETRIC_REGISTERED"
    applications: ["VOTA", "*"]
    regex: "(?i)^(?!(?:.*\\b(?:mes[aá]rio|presidente)\\b))(?:.*(?:n[aã]o\\s+h[aá]\\s+biometria(?:\\s+cadastrada)?|eleitor\\s+(?:n[aã]o\\s+possui|sem)\\s+biometria(?:\\s+cadastrada)?|aus[eê]ncia\\s+de\\s+biometria\\s+no\\s+cadastro|habilita[cç][aã]o\\s+de\\s+eleitor\\s+sem\\s+biometria|ano\\s+de\\s+nascimento\\s+digitado\\s+[eé]\\s+igual\\s+ao\\s+do\\s+cadastro))"
    extractors: {}
    description: "Eleitor sem biometria cadastrada no banco de dados da seção (incluindo habilitação por ano de nascimento)."

  - pattern_id: "manual_enable_path_main"
    event_code: "MANUAL_ENABLE_PATH"
    applications: ["VOTA", "*"]
    regex: "(?i)(?:habilita[cç][aã]o\\s+manual(?:\\s+pelo\\s+mes[aá]rio)?|libera[cç][aã]o\\s+manual(?:\\s+pelo\\s+mes[aá]rio)?|eleitor\\s+habilitado\\s+(?:manualmente|sem\\s+biometria)|habilitado\\s+sem\\s+biometria\\s+ap[oó]s\\s+tentativas|identifica[cç][aã]o\\s+manual\\s+do\\s+eleitor|c[oó]digo\\s+de\\s+habilita[cç][aã]o\\s+digitado\\s+pelo\\s+mes[aá]rio|solicita[cç][aã]o\\s+de\\s+dado\\s+pessoal\\s+do\\s+eleitor\\s+para\\s+habilita[cç][aã]o\\s+manual|mes[aá]rio\\s+\\d+\\s+habilitou\\s+o\\s+eleitor)"
    extractors: {}
    description: "Habilitação manual pelo mesário após tentativas biométricas malsucedidas."

  - pattern_id: "inactivity_timeout_45s_main"
    event_code: "INACTIVITY_TIMEOUT_45S"
    applications: ["VOTA", "*"]
    regex: "(?i)(?:eleitor\\s+sem\\s+atividade\\s+por\\s+45\\s*s(?:egundos)?|sem\\s+atividade\\s+por\\s+45\\s*s(?:egundos)?|tempo\\s+limite\\s+de\\s+inatividade|timeout\\s+de\\s+inatividade|inatividade\\s+(?:de|por)\\s+45\\s*s(?:egundos)?|cancelamento\\s+por\\s+inatividade(?:\\s+de\\s+45\\s*s(?:egundos)?)?)"
    extractors: {}
    description: "Alerta ou cancelamento por inatividade de 45 segundos durante a sessão (incluindo formato oficial TSE 'Eleitor sem atividade por 45 segundos')."

  - pattern_id: "invalid_key_pressed_main"
    event_code: "INVALID_KEY_PRESSED"
    applications: ["VOTA", "*"]
    regex: "(?i)(?:acionamento\\s+de\\s+)?tecla\\s+(?:indevida|inv[aá]lida)(?:\\s+pressionada)?"
    extractors: {}
    description: "Acionamento de tecla indevida ou inválida durante a habilitação ou votação."

  - pattern_id: "cargo_vote_recorded_main"
    event_code: "CARGO_VOTE_RECORDED"
    applications: ["VOTA", "*"]
    regex: "(?i)(?:voto\\s+(?:confirmado|registrado)\\s+para\\b|confirma[cç][aã]o\\s+de\\s+voto\\s+no\\s+cargo)"
    extractors: {}
    description: "Confirmação de voto em um cargo individual sem identificação do votável (ex.: 'Voto confirmado para [...]')."

  - pattern_id: "voter_vote_computed_main"
    event_code: "VOTER_VOTE_COMPUTED"
    applications: ["VOTA", "*"]
    regex: "(?i)(?:o\\s+)?voto\\s+do\\s+eleitor\\s+(?:foi\\s+)?computado"
    extractors: {}
    description: "Conclusão e computação final do voto do eleitor na seção."

  - pattern_id: "voting_suspended_main"
    event_code: "VOTING_SUSPENDED"
    applications: ["VOTA", "GAP", "*"]
    regex: "(?i)(?:vota[cç][aã]o\\s+suspensa|suspens[aã]o\\s+(?:tempor[aá]ria\\s+|parcial\\s+|total\\s+)?da\\s+vota[cç][aã]o|eleitor\\s+suspenso\\s+pelo\\s+mes[aá]rio)"
    extractors: {}
    description: "Suspensão temporária, parcial ou total da votação na seção eleitoral."

  - pattern_id: "voting_resumed_main"
    event_code: "VOTING_RESUMED"
    applications: ["VOTA", "GAP", "*"]
    regex: "(?i)(?:vota[cç][aã]o\\s+retomada|retomada\\s+da\\s+vota[cç][aã]o)"
    extractors: {}
    description: "Retomada da votação após suspensão ou procedimento técnico."

  - pattern_id: "contingency_event_main"
    event_code: "CONTINGENCY_EVENT"
    applications: ["VOTA", "GAP", "RED", "scue", "adm", "*"]
    regex: "(?i)(?:modo\\s+de\\s+conting[eê]ncia|substitui[cç][aã]o\\s+de\\s+urna|recupera[cç][aã]o\\s+de\\s+dados\\s+de\\s+conting[eê]ncia|carga\\s+de\\s+conting[eê]ncia|in[ií]cio\\s+de\\s+conting[eê]ncia)"
    extractors: {}
    description: "Operação técnica de contingência, carga ou substituição de urna."

  - pattern_id: "section_closed_main"
    event_code: "SECTION_CLOSED"
    applications: ["VOTA", "GAP", "RED", "*"]
    regex: "(?i)(?:encerramento\\s+da\\s+(?:vota[cç][aã]o|se[cç][aã]o)|vota[cç][aã]o\\s+encerrada|se[cç][aã]o\\s+encerrada|emiss[aã]o\\s+do\\s+boletim\\s+de\\s+urna|gerando\\s+arquivo\\s+de\\s+resultado\\s*\\[imgbu\\.dat\\]\\s*\\+\\s*\\[t[eé]rmino\\])"
    extractors: {}
    description: "Encerramento oficial da votação na seção eleitoral e emissão do BU."

turn_boundary_patterns:
  turn1_init: "(?i)(?:iniciando\\s+aplica[cç][aã]o\\s*[-–:]\\s*1[º°o]?\\s*turno|turno(?:\\s+da\\s+ue)?\\s*[:=\\-]\\s*\\[?1\\]?(?:\\D|$)|\\b1[º°o]\\s*turno\\b|\\bprimeiro\\s+turno\\b|\\bpleito\\s*[:=\\-]?\\s*\\[?0?3220\\]?(?:\\D|$)|\\bo03220[a-z]{2})"
  turn2_init: "(?i)(?:iniciando\\s+aplica[cç][aã]o\\s*[-–:]\\s*2[º°o]?\\s*turno|turno(?:\\s+da\\s+ue)?\\s*[:=\\-]\\s*\\[?2\\]?(?:\\D|$)|\\b2[º°o]\\s*turno\\b|\\bsegundo\\s+turno\\b|\\bpleito\\s*[:=\\-]?\\s*\\[?0?3221\\]?(?:\\D|$)|\\bo03221[a-z]{2})"
  boot_header: "(?i)(?:in[ií]cio\\s+das\\s+opera[cç][oõ]es\\s+do\\s+logd|logd\\s+iniciado)"
  turn1_close: "(?i)(?:encerramento\\s+da\\s+(?:vota[cç][aã]o|se[cç][aã]o)|vota[cç][aã]o\\s+encerrada|se[cç][aã]o\\s+encerrada|emiss[aã]o\\s+do\\s+boletim\\s+de\\s+urna|gerando\\s+arquivo\\s+de\\s+resultado\\s*\\[(?:imgbu\\.dat|log\\.jez)\\])"
"""

# ---------------------------------------------------------------------------
# Dataclasses e funções inline (sem import tse_analysis)
# ---------------------------------------------------------------------------
from dataclasses import dataclass
from dataclasses import field as dc_field
from typing import Any

UNKNOWN_EVENT_CODE: str = "UNKNOWN_EVENT"
MALFORMED_LINE_CODE: str = "MALFORMED_LINE"
ALLOWED_EXTRACTOR_FIELDS: tuple = ("biometric_attempt_num",)
FORBIDDEN_EXTRACTOR_TOKENS: tuple = (
    "cpf", "titulo", "eleitor", "mesario", "message",
    "text", "mac", "voto", "votavel", "candidato",
)

_DEFAULT_TURN_BOUNDARY_RAW: dict[str, str] = {
    "turn1_init": (
        r"(?i)(?:iniciando\s+aplica[cç][aã]o\s*[-–:]\s*1[º°o]?\s*turno"
        r"|turno(?:\s+da\s+ue)?\s*[:=\-]\s*\[?1\]?(?:\D|$)"
        r"|\b1[º°o]\s*turno\b|\bprimeiro\s+turno\b"
        r"|\bpleito\s*[:=\-]?\s*\[?0?3220\]?(?:\D|$)|\bo03220[a-z]{2})"
    ),
    "turn2_init": (
        r"(?i)(?:iniciando\s+aplica[cç][aã]o\s*[-–:]\s*2[º°o]?\s*turno"
        r"|turno(?:\s+da\s+ue)?\s*[:=\-]\s*\[?2\]?(?:\D|$)"
        r"|\b2[º°o]\s*turno\b|\bsegundo\s+turno\b"
        r"|\bpleito\s*[:=\-]?\s*\[?0?3221\]?(?:\D|$)|\bo03221[a-z]{2})"
    ),
    "boot_header": r"(?i)(?:in[ií]cio\s+das\s+opera[cç][oõ]es\s+do\s+logd|logd\s+iniciado)",
    "turn1_close": (
        r"(?i)(?:encerramento\s+da\s+(?:vota[cç][aã]o|se[cç][aã]o)"
        r"|vota[cç][aã]o\s+encerrada|se[cç][aã]o\s+encerrada"
        r"|emiss[aã]o\s+do\s+boletim\s+de\s+urna"
        r"|gerando\s+arquivo\s+de\s+resultado\s*\[(?:imgbu\.dat|log\.jez)\])"
    ),
}


class EventCatalogValidationError(ValueError):
    pass


@dataclass(frozen=True)
class ClassificationResult:
    event_code: str
    biometric_attempt_num: int | None = None
    pattern_id: str | None = None


@dataclass(frozen=True)
class EventPattern:
    pattern_id: str
    event_code: str
    applications: tuple
    regex: str
    compiled_regex: re.Pattern
    extractors: dict = dc_field(default_factory=dict)
    description: str = ""

    def matches_application(self, application: str) -> bool:
        if "*" in self.applications:
            return True
        normalized_app = application.strip().upper()
        return any(app.upper() == normalized_app for app in self.applications)


@dataclass(frozen=True)
class TurnBoundaryPatterns:
    turn1_init: str
    turn2_init: str
    boot_header: str
    turn1_close: str
    compiled_turn1_init: re.Pattern
    compiled_turn2_init: re.Pattern
    compiled_boot_header: re.Pattern
    compiled_turn1_close: re.Pattern


def _validate_and_build_turn_boundary_patterns(raw: Any) -> TurnBoundaryPatterns:
    if raw is None:
        raw_map = dict(_DEFAULT_TURN_BOUNDARY_RAW)
    elif isinstance(raw, dict):
        raw_map = {**_DEFAULT_TURN_BOUNDARY_RAW, **{str(k): str(v) for k, v in raw.items()}}
    else:
        raise EventCatalogValidationError("Campo 'turn_boundary_patterns' deve ser um mapeamento YAML.")

    compiled: dict[str, re.Pattern] = {}
    patterns: dict[str, str] = {}
    for key in ("turn1_init", "turn2_init", "boot_header", "turn1_close"):
        regex_str = str(raw_map.get(key, "")).strip()
        if not regex_str:
            raise EventCatalogValidationError(f"Padrão '{key}' em 'turn_boundary_patterns' não pode ser vazio.")
        compiled[key] = re.compile(regex_str)
        patterns[key] = regex_str

    return TurnBoundaryPatterns(
        turn1_init=patterns["turn1_init"],
        turn2_init=patterns["turn2_init"],
        boot_header=patterns["boot_header"],
        turn1_close=patterns["turn1_close"],
        compiled_turn1_init=compiled["turn1_init"],
        compiled_turn2_init=compiled["turn2_init"],
        compiled_boot_header=compiled["boot_header"],
        compiled_turn1_close=compiled["turn1_close"],
    )


@dataclass(frozen=True)
class EventCatalog:
    schema_version: str
    fallback_event_code: str
    malformed_event_code: str
    patterns: tuple
    turn_boundary_patterns: TurnBoundaryPatterns = dc_field(
        default_factory=lambda: _validate_and_build_turn_boundary_patterns(None)
    )

    @property
    def event_codes(self) -> frozenset:
        return frozenset(p.event_code for p in self.patterns)

    def classify(self, message_text: str, application: str = "VOTA") -> ClassificationResult:
        normalized_msg = message_text.strip()
        if not normalized_msg:
            return ClassificationResult(
                event_code=self.fallback_event_code,
                biometric_attempt_num=None,
                pattern_id=None,
            )
        for pattern in self.patterns:
            if not pattern.matches_application(application):
                continue
            match = pattern.compiled_regex.search(normalized_msg)
            if match is None:
                continue
            attempt_num: int | None = None
            group_name = pattern.extractors.get("biometric_attempt_num")
            if group_name:
                raw_val = match.groupdict().get(group_name)
                if raw_val is not None:
                    try:
                        parsed_int = int(raw_val)
                        if 1 <= parsed_int <= 127:
                            attempt_num = parsed_int
                    except ValueError:
                        attempt_num = None
            return ClassificationResult(
                event_code=pattern.event_code,
                biometric_attempt_num=attempt_num,
                pattern_id=pattern.pattern_id,
            )
        return ClassificationResult(
            event_code=self.fallback_event_code,
            biometric_attempt_num=None,
            pattern_id=None,
        )


def _validate_and_build_pattern(raw: Any, seen_pattern_ids: set) -> EventPattern:
    if not isinstance(raw, dict):
        raise EventCatalogValidationError("Cada item em 'patterns' deve ser um objeto YAML.")
    for req_field in ("pattern_id", "event_code", "applications", "regex"):
        if req_field not in raw or raw[req_field] is None:
            raise EventCatalogValidationError(
                f"Regra de evento inválida: campo obrigatório ausente '{req_field}'."
            )
    pattern_id = str(raw["pattern_id"]).strip()
    if not pattern_id:
        raise EventCatalogValidationError("Campo 'pattern_id' não pode ser vazio.")
    if pattern_id in seen_pattern_ids:
        raise EventCatalogValidationError(f"pattern_id duplicado no catálogo: '{pattern_id}'.")
    seen_pattern_ids.add(pattern_id)

    event_code = str(raw["event_code"]).strip()
    if not event_code or not re.fullmatch(r"[A-Z][A-Z0-9_]*", event_code):
        raise EventCatalogValidationError(
            f"Regra '{pattern_id}' possui event_code inválido '{event_code}'. Use UPPER_SNAKE_CASE."
        )
    apps_raw = raw["applications"]
    if not isinstance(apps_raw, list) or not apps_raw:
        raise EventCatalogValidationError(
            f"Regra '{pattern_id}' deve definir 'applications' como lista não-vazia."
        )
    applications = tuple(str(app).strip() for app in apps_raw if str(app).strip())

    regex_str = str(raw["regex"])
    if not regex_str.strip():
        raise EventCatalogValidationError(f"Regra '{pattern_id}' possui regex vazia.")
    try:
        compiled = re.compile(regex_str)
    except re.error as exc:
        raise EventCatalogValidationError(
            f"Expressão regular inválida na regra '{pattern_id}': {exc}"
        ) from exc

    for group_name in compiled.groupindex:
        lowered_group = group_name.lower()
        if any(token in lowered_group for token in FORBIDDEN_EXTRACTOR_TOKENS):
            raise EventCatalogValidationError(
                f"Regra '{pattern_id}' declara grupo regex proibido '{group_name}' "
                "(violação de política zero-PII/zero-texto-livre)."
            )

    extractors_raw = raw.get("extractors") or {}
    if not isinstance(extractors_raw, dict):
        raise EventCatalogValidationError(
            f"Campo 'extractors' na regra '{pattern_id}' deve ser um mapeamento."
        )
    extractors: dict = {}
    for target_field, source_group in extractors_raw.items():
        field_str = str(target_field).strip()
        group_str = str(source_group).strip()
        lowered_field = field_str.lower()
        if (
            any(token in lowered_field for token in FORBIDDEN_EXTRACTOR_TOKENS)
            or field_str not in ALLOWED_EXTRACTOR_FIELDS
        ):
            raise EventCatalogValidationError(
                f"Regra '{pattern_id}' tenta extrair campo não permitido '{field_str}'."
            )
        if group_str not in compiled.groupindex:
            raise EventCatalogValidationError(
                f"Regra '{pattern_id}' referencia grupo regex inexistente '{group_str}'."
            )
        extractors[field_str] = group_str

    return EventPattern(
        pattern_id=pattern_id,
        event_code=event_code,
        applications=applications,
        regex=regex_str,
        compiled_regex=compiled,
        extractors=extractors,
        description=str(raw.get("description", "")).strip(),
    )


def load_event_catalog_from_string(yaml_str: str) -> EventCatalog:
    """Carrega, compila e valida o catálogo a partir de uma string YAML inline."""
    payload = yaml.safe_load(yaml_str)
    if not isinstance(payload, dict):
        raise EventCatalogValidationError("O conteúdo YAML deve ser um objeto raiz.")
    raw_patterns = payload.get("patterns")
    if not isinstance(raw_patterns, list) or not raw_patterns:
        raise EventCatalogValidationError(
            "O catálogo deve conter uma lista não-vazia em 'patterns'."
        )
    seen_ids: set = set()
    patterns = tuple(_validate_and_build_pattern(item, seen_ids) for item in raw_patterns)
    turn_boundary_patterns = _validate_and_build_turn_boundary_patterns(
        payload.get("turn_boundary_patterns")
    )
    return EventCatalog(
        schema_version=str(payload.get("schema_version", "1.0")),
        fallback_event_code=str(payload.get("fallback_event_code", UNKNOWN_EVENT_CODE)),
        malformed_event_code=str(payload.get("malformed_event_code", MALFORMED_LINE_CODE)),
        patterns=patterns,
        turn_boundary_patterns=turn_boundary_patterns,
    )


CATALOG = load_event_catalog_from_string(EVENT_CATALOG_YAML)
print(f"Catálogo carregado: {len(CATALOG.patterns)} padrões + turn_boundary_patterns")

In [ ]:
# ---------------------------------------------------------------------------
# Silver schema definition (17 colunas — inclui is_clock_drift)
# ---------------------------------------------------------------------------
SILVER_SCHEMA = pl.Schema({
    "election_year": pl.Int16,
    "turn": pl.Int8,
    "uf": pl.String,
    "municipio_code": pl.Int32,
    "zone_num": pl.Int16,
    "section_num": pl.Int32,
    "urna_serial": pl.String,
    "file_seq": pl.Int64,
    "raw_timestamp": pl.Datetime("us"),
    "is_clock_drift": pl.Boolean,
    "has_timestamp_regression": pl.Boolean,
    "application": pl.String,
    "log_level": pl.String,
    "event_code": pl.String,
    "biometric_attempt_num": pl.Int8,
    "classification_status": pl.String,
    "parser_version": pl.String,
})

print("Silver schema (17 colunas):")
for col_name, col_type in SILVER_SCHEMA.items():
    print(f"  {col_name}: {col_type}")

In [ ]:
from collections.abc import Iterable
from dataclasses import dataclass
from datetime import datetime as dt_type
from pathlib import Path

import polars as pl


# ---------------------------------------------------------------------------
# classify_row + Turn 2 carryover stripping (clock-drift-safe, vetorizado)
# ---------------------------------------------------------------------------
def classify_row(message_text: str, application: str, catalog: EventCatalog) -> dict:
    """Classifica uma linha de log e retorna dict com event_code, biometric_attempt_num, classification_status."""
    result = catalog.classify(message_text, application)
    is_classified = result.event_code not in (
        catalog.fallback_event_code,
        catalog.malformed_event_code,
    )
    return {
        "event_code": result.event_code,
        "biometric_attempt_num": result.biometric_attempt_num,
        "classification_status": "CLASSIFIED" if is_classified else "UNKNOWN",
    }





SILVER_PARSER_VERSION: str = "1.1.0"

RESOLUTION_METHODS: tuple[str, ...] = (
    "lifecycle_marker",
    "turn1_urna_seq_overlap",
    "date_fallback",
    "fresh_urna_no_carryover",
    "unresolved",
)

SILVER_FACT_URNA_EVENT_SCHEMA = pl.Schema(
    {
        "election_year": pl.Int16,
        "turn": pl.Int8,
        "uf": pl.String,
        "municipio_code": pl.Int32,
        "zone_num": pl.Int16,
        "section_num": pl.Int32,
        "urna_serial": pl.String,
        "file_seq": pl.Int64,
        "raw_timestamp": pl.Datetime("us"),
        "is_clock_drift": pl.Boolean,
        "has_timestamp_regression": pl.Boolean,
        "application": pl.String,
        "log_level": pl.String,
        "event_code": pl.String,
        "biometric_attempt_num": pl.Int8,
        "classification_status": pl.String,
        "parser_version": pl.String,
    }
)

TURN1_HISTORY_SCHEMA = pl.Schema(
    {
        "urna_serial": pl.String,
        "t1_max_file_seq": pl.Int64,
        "t1_first_ts_raw": pl.String,
        "t1_max_seq_ts_raw": pl.String,
    }
)

_SESSION_ACTIVITY_REGEX: str = (
    r"(?i)(?:urna\s+pronta\s+para\s+receber\s+votos"
    r"|in[ií]cio\s+da\s+vota[cç][aã]o"
    r"|emiss[aã]o\s+d[ae]\s+zer[eé]sima"
    r"|eleitor\s+(?:foi\s+)?habilitado"
    r"|voto\s+do\s+eleitor"
    r"|modo\s+de\s+conting[eê]ncia)"
)


@dataclass(frozen=True, slots=True)
class CarryoverFilterStats:
    """Estatísticas de auditoria da remoção de carryover do 1º turno em um lote/partição."""

    raw_rows: int
    carryover_rows_dropped: int
    silver_rows_kept: int
    clock_drift_rows_retained: int
    jez_files_total: int
    jez_resolution_counts: dict[str, int]


def _empty_resolution_counts() -> dict[str, int]:
    return {method: 0 for method in RESOLUTION_METHODS}


def default_inter_turn_cutoff(election_year: int) -> dt_type:
    """Retorna a data de corte padrão entre 1º e 2º turno para fallback temporal."""
    return dt_type(election_year, 10, 10, 0, 0, 0)


def election_clock_window(election_year: int) -> tuple[dt_type, dt_type]:
    """Janela operacional válida para detecção de clock drift da BIOS da urna."""
    return (
        dt_type(election_year, 8, 1, 0, 0, 0),
        dt_type(election_year, 12, 31, 23, 59, 59, 999999),
    )


def parse_bronze_timestamps_expr(
    col_name: str = "raw_timestamp", *, dtype: pl.DataType | None = None
) -> pl.Expr:
    """Expressão Polars vetorizada para converter raw_timestamp (string ou datetime) em Datetime('us')."""
    if dtype is not None and isinstance(dtype, pl.Datetime):
        return pl.col(col_name).cast(pl.Datetime("us"))

    cleaned = pl.col(col_name).cast(pl.String).str.strip_chars().str.replace(",", ".", literal=True)
    return pl.coalesce(
        [
            cleaned.str.to_datetime("%d/%m/%Y %H:%M:%S", time_unit="us", strict=False),
            cleaned.str.to_datetime("%d/%m/%Y %H:%M:%S%.f", time_unit="us", strict=False),
            cleaned.str.to_datetime("%Y-%m-%d %H:%M:%S", time_unit="us", strict=False),
            cleaned.str.to_datetime("%Y-%m-%d %H:%M:%S%.f", time_unit="us", strict=False),
        ]
    )


def compute_clock_drift_expr(ts_expr: pl.Expr, election_year: int) -> pl.Expr:
    """Flag booleana True quando o timestamp válido está fora da janela eleitoral esperada."""
    win_start, win_end = election_clock_window(election_year)
    return (
        ts_expr.is_not_null() & ((ts_expr < pl.lit(win_start)) | (ts_expr > pl.lit(win_end)))
    ).fill_null(False)


def _jez_key_expr(columns: Iterable[str]) -> pl.Expr:
    cols = set(columns)
    if "source_jez_file" in cols:
        return pl.col("source_jez_file").cast(pl.String).fill_null("UNKNOWN_JEZ")

    mun_col = "cd_municipio" if "cd_municipio" in cols else "municipio_code"
    zone_col = "nr_zona" if "nr_zona" in cols else "zone_num"
    sec_col = "nr_secao" if "nr_secao" in cols else "section_num"
    parts = [
        pl.col("uf").cast(pl.String).fill_null("") if "uf" in cols else pl.lit(""),
        pl.col(mun_col).cast(pl.String).fill_null("") if mun_col in cols else pl.lit(""),
        pl.col(zone_col).cast(pl.String).fill_null("") if zone_col in cols else pl.lit(""),
        pl.col(sec_col).cast(pl.String).fill_null("") if sec_col in cols else pl.lit(""),
        pl.col("urna_serial").cast(pl.String).fill_null("")
        if "urna_serial" in cols
        else pl.lit(""),
    ]
    return pl.concat_str(parts, separator=":")


def extract_turn1_shard_history(df_turn1: pl.DataFrame) -> pl.DataFrame:
    """Extrai resumo compacto (urna_serial, t1_max_file_seq, timestamps) de um shard de 1º turno."""
    if df_turn1.is_empty():
        return pl.DataFrame(schema=TURN1_HISTORY_SCHEMA)

    valid = df_turn1.filter(
        pl.col("urna_serial").is_not_null()
        & (pl.col("urna_serial").cast(pl.String).str.strip_chars() != "")
        & (pl.col("urna_serial").cast(pl.String).str.strip_chars() != "UNKNOWN")
    )
    if valid.is_empty():
        return pl.DataFrame(schema=TURN1_HISTORY_SCHEMA)

    return (
        valid.with_columns(
            pl.col("urna_serial").cast(pl.String).str.strip_chars().alias("urna_serial"),
            pl.col("file_seq").cast(pl.Int64).alias("file_seq"),
            pl.col("raw_timestamp").cast(pl.String).alias("_raw_ts_str"),
        )
        .group_by("urna_serial")
        .agg(
            pl.col("file_seq").max().cast(pl.Int64).alias("t1_max_file_seq"),
            pl.col("_raw_ts_str").sort_by("file_seq").first().alias("t1_first_ts_raw"),
            pl.col("_raw_ts_str").sort_by("file_seq").last().alias("t1_max_seq_ts_raw"),
        )
        .select(list(TURN1_HISTORY_SCHEMA.names()))
    )


def merge_turn1_history(
    existing: pl.DataFrame | None,
    new_chunk: pl.DataFrame | None,
) -> pl.DataFrame:
    """Combina dois resumos de histórico de 1º turno mantendo o maior t1_max_file_seq por urna."""
    frames = [f for f in (existing, new_chunk) if f is not None and not f.is_empty()]
    if not frames:
        return pl.DataFrame(schema=TURN1_HISTORY_SCHEMA)

    normalized: list[pl.DataFrame] = []
    for frame in frames:
        exprs: list[pl.Expr] = [
            pl.col("urna_serial").cast(pl.String),
            pl.col("t1_max_file_seq").cast(pl.Int64),
        ]
        if "t1_first_ts_raw" in frame.columns:
            exprs.append(pl.col("t1_first_ts_raw").cast(pl.String))
        else:
            exprs.append(pl.lit(None, dtype=pl.String).alias("t1_first_ts_raw"))
        if "t1_max_seq_ts_raw" in frame.columns:
            exprs.append(pl.col("t1_max_seq_ts_raw").cast(pl.String))
        else:
            exprs.append(pl.lit(None, dtype=pl.String).alias("t1_max_seq_ts_raw"))
        normalized.append(frame.select(exprs))

    combined = pl.concat(normalized, how="vertical")
    return (
        combined.sort("t1_max_file_seq", descending=True)
        .unique(subset=["urna_serial"], keep="first")
        .select(list(TURN1_HISTORY_SCHEMA.names()))
    )


def scan_turn1_urna_history(parquet_files: Iterable[str | Path]) -> pl.DataFrame:
    """Constrói histórico compacto de 1º turno em streaming lendo apenas 3 colunas por shard."""
    history: pl.DataFrame | None = None
    for pq_path in parquet_files:
        shard_df = pl.read_parquet(pq_path, columns=["urna_serial", "file_seq", "raw_timestamp"])
        chunk_hist = extract_turn1_shard_history(shard_df)
        if not chunk_hist.is_empty():
            history = merge_turn1_history(history, chunk_hist)
        del shard_df, chunk_hist
    return history if history is not None else pl.DataFrame(schema=TURN1_HISTORY_SCHEMA)


def _normalize_turn1_history(turn1_history: pl.DataFrame | None) -> pl.DataFrame | None:
    if turn1_history is None or turn1_history.is_empty():
        return None
    return merge_turn1_history(None, turn1_history)


def strip_turn1_carryover(
    df: pl.DataFrame,
    *,
    election_year: int = 2026,
    turn: int,
    catalog: EventCatalog | None = None,
    turn1_history: pl.DataFrame | None = None,
    inter_turn_cutoff: dt_type | None = None,
) -> tuple[pl.DataFrame, CarryoverFilterStats]:
    """Remove prefixo de carryover do 1º turno apenas quando turn == 2, preservando clock drift.

    Mantém 100% das linhas quando turn == 1. Quando turn == 2, resolve o início do 2º turno
    por arquivo `.jez` usando:
      1) Marcadores estruturais/ciclo de vida (turn2_init + boot_header) via `file_seq`
      2) Sobreposição de histórico do 1º turno por `urna_serial` (`t1_max_file_seq`)
      3) Fallback de data apenas sobre timestamps válidos sem `is_clock_drift`
    """
    raw_rows = df.height
    if raw_rows == 0:
        out_empty = (
            df
            if "is_clock_drift" in df.columns
            else df.with_columns(pl.lit(False).alias("is_clock_drift"))
        )
        return out_empty, CarryoverFilterStats(
            raw_rows=0,
            carryover_rows_dropped=0,
            silver_rows_kept=0,
            clock_drift_rows_retained=0,
            jez_files_total=0,
            jez_resolution_counts=_empty_resolution_counts(),
        )

    ts_dtype = df.schema.get("raw_timestamp")
    parsed_ts_expr = parse_bronze_timestamps_expr("raw_timestamp", dtype=ts_dtype)
    drift_expr = compute_clock_drift_expr(parsed_ts_expr, election_year)

    if turn != 2:
        out_t1 = (
            df
            if "is_clock_drift" in df.columns
            else df.with_columns(drift_expr.alias("is_clock_drift"))
        )
        drift_cnt = int(out_t1.select(pl.col("is_clock_drift").sum()).item() or 0)
        jez_total = int(out_t1.select(_jez_key_expr(out_t1.columns).n_unique()).item() or 0)
        return out_t1, CarryoverFilterStats(
            raw_rows=raw_rows,
            carryover_rows_dropped=0,
            silver_rows_kept=raw_rows,
            clock_drift_rows_retained=drift_cnt,
            jez_files_total=jez_total,
            jez_resolution_counts=_empty_resolution_counts(),
        )

    active_catalog = catalog if catalog is not None else CATALOG
    tbp = active_catalog.turn_boundary_patterns
    cutoff_dt = inter_turn_cutoff or default_inter_turn_cutoff(election_year)
    norm_hist = _normalize_turn1_history(turn1_history)
    has_history_source = norm_hist is not None

    has_msg = "message_text" in df.columns
    has_event_code = "event_code" in df.columns

    # Projeção mínima para resolução de fronteira (evita duplicar message_text e demais colunas em RAM)
    meta_exprs: list[pl.Expr] = [
        _jez_key_expr(df.columns).alias("_jez_key"),
        pl.col("urna_serial")
        .cast(pl.String)
        .str.strip_chars()
        .fill_null("UNKNOWN")
        .alias("_serial_norm"),
        pl.col("file_seq").cast(pl.Int64).alias("_file_seq_i64"),
        pl.col("raw_timestamp").cast(pl.String).alias("_raw_ts_str"),
        parsed_ts_expr.alias("_parsed_ts"),
        drift_expr.alias("_is_clock_drift"),
    ]

    if has_msg:
        msg_str = pl.col("message_text").cast(pl.String).fill_null("")
        meta_exprs.extend(
            [
                msg_str.str.contains(tbp.turn1_init).alias("_is_t1_init"),
                msg_str.str.contains(tbp.turn2_init).alias("_is_t2_init"),
                msg_str.str.contains(tbp.boot_header).alias("_is_boot"),
                msg_str.str.contains(tbp.turn1_close).alias("_is_t1_close"),
                msg_str.str.contains(_SESSION_ACTIVITY_REGEX).alias("_is_activity"),
            ]
        )
    elif has_event_code:
        ev_str = pl.col("event_code").cast(pl.String).fill_null("")
        meta_exprs.extend(
            [
                pl.lit(False).alias("_is_t1_init"),
                pl.lit(False).alias("_is_t2_init"),
                (ev_str == "LOGD_STARTED").alias("_is_boot"),
                (ev_str == "SECTION_CLOSED").alias("_is_t1_close"),
                ev_str.is_in(
                    ["SECTION_OPENED", "VOTER_ENABLED", "VOTER_VOTE_COMPUTED", "CONTINGENCY_EVENT"]
                ).alias("_is_activity"),
            ]
        )
    else:
        meta_exprs.extend(
            [
                pl.lit(False).alias("_is_t1_init"),
                pl.lit(False).alias("_is_t2_init"),
                pl.lit(False).alias("_is_boot"),
                pl.lit(False).alias("_is_t1_close"),
                pl.lit(False).alias("_is_activity"),
            ]
        )

    work = df.select(meta_exprs)

    if norm_hist is not None:
        work = work.join(
            norm_hist.rename({"urna_serial": "_serial_norm"}),
            on="_serial_norm",
            how="left",
        )
    else:
        work = work.with_columns(
            pl.lit(None, dtype=pl.Int64).alias("t1_max_file_seq"),
            pl.lit(None, dtype=pl.String).alias("t1_first_ts_raw"),
            pl.lit(None, dtype=pl.String).alias("t1_max_seq_ts_raw"),
        )

    non_drifted_ts = pl.when(pl.col("_parsed_ts").is_not_null() & ~pl.col("_is_clock_drift")).then(
        pl.col("_parsed_ts")
    )

    per_jez = work.group_by("_jez_key").agg(
        pl.col("_file_seq_i64").min().alias("_min_seq"),
        pl.col("_file_seq_i64").max().alias("_max_seq"),
        pl.col("_raw_ts_str").sort_by("_file_seq_i64").first().alias("_first_ts_raw"),
        pl.col("t1_max_file_seq").first().alias("_t1_max_seq"),
        pl.col("t1_first_ts_raw").first().alias("_t1_first_ts"),
        pl.col("t1_max_seq_ts_raw").first().alias("_t1_max_seq_ts"),
        pl.col("_raw_ts_str")
        .filter(pl.col("_file_seq_i64") == pl.col("t1_max_file_seq"))
        .first()
        .alias("_ts_at_t1_max"),
        pl.col("_is_t2_init")
        .filter(pl.col("_file_seq_i64") <= pl.col("t1_max_file_seq"))
        .any()
        .alias("_t2_init_le_t1_max"),
        pl.col("_is_t1_init").any().alias("_any_t1_init"),
        pl.col("_file_seq_i64").filter(pl.col("_is_t2_init")).min().alias("_any_t2_init_seq"),
        pl.col("_file_seq_i64")
        .filter(pl.col("_is_t2_init") & (pl.col("_file_seq_i64") > pl.col("t1_max_file_seq")))
        .min()
        .alias("_t2_init_after_t1_seq"),
        (non_drifted_ts < pl.lit(cutoff_dt)).any().fill_null(False).alias("_has_pre_cutoff"),
        (non_drifted_ts >= pl.lit(cutoff_dt)).any().fill_null(False).alias("_has_post_cutoff"),
        pl.col("_file_seq_i64")
        .filter(
            pl.col("_parsed_ts").is_not_null()
            & ~pl.col("_is_clock_drift")
            & (pl.col("_parsed_ts") < pl.lit(cutoff_dt))
        )
        .max()
        .alias("_last_pre_ts_seq"),
        pl.col("_file_seq_i64")
        .filter(
            pl.col("_parsed_ts").is_not_null()
            & ~pl.col("_is_clock_drift")
            & (pl.col("_parsed_ts") >= pl.lit(cutoff_dt))
        )
        .min()
        .alias("_first_post_ts_seq"),
    )

    # Identifica sobreposição real de Turno 1 vs Turno 2 (protegendo contra flash formatada)
    ts_fingerprint_present = (
        pl.col("_t1_first_ts").is_not_null() | pl.col("_t1_max_seq_ts").is_not_null()
    )
    ts_fingerprint_match = (pl.col("_first_ts_raw") == pl.col("_t1_first_ts")).fill_null(False) | (
        pl.col("_ts_at_t1_max") == pl.col("_t1_max_seq_ts")
    ).fill_null(False)
    per_jez = per_jez.with_columns(
        (pl.col("_t1_max_seq").is_not_null() & (pl.col("_min_seq") > pl.col("_t1_max_seq"))).alias(
            "_already_past_t1"
        ),
        (
            pl.col("_t1_max_seq").is_not_null()
            & (pl.col("_min_seq") <= pl.col("_t1_max_seq"))
            & (pl.col("_max_seq") > pl.col("_t1_max_seq"))
            & pl.when(ts_fingerprint_present)
            .then(ts_fingerprint_match)
            .otherwise(~pl.col("_t2_init_le_t1_max").fill_null(False))
        ).alias("_has_t1_overlap"),
    ).with_columns(
        pl.when(pl.col("_has_t1_overlap"))
        .then(pl.col("_t2_init_after_t1_seq"))
        .otherwise(pl.col("_any_t2_init_seq"))
        .alias("_first_t2_init_seq")
    )

    # 2º passe compacto por _jez_key para ancorar no último evento de T1 e primeiro boot de T2
    anchor_cols = [
        "_jez_key",
        "_file_seq_i64",
        "_is_t1_close",
        "_is_t1_init",
        "_is_boot",
        "_is_activity",
        "t1_max_file_seq",
    ]
    work_thin = work.select(anchor_cols)
    drift_series = work.get_column("_is_clock_drift")
    jez_key_series = work.get_column("_jez_key")
    file_seq_series = work.get_column("_file_seq_i64")
    del work

    work_anchors = work_thin.join(
        per_jez.select(["_jez_key", "_first_t2_init_seq", "_first_post_ts_seq"]),
        on="_jez_key",
        how="left",
    )

    per_jez_anchors = work_anchors.group_by("_jez_key").agg(
        pl.col("_file_seq_i64")
        .filter(pl.col("_is_t1_close") & (pl.col("_file_seq_i64") < pl.col("_first_t2_init_seq")))
        .max()
        .alias("_last_t1_close_before_t2"),
        pl.col("_file_seq_i64")
        .filter(pl.col("_is_t1_init") & (pl.col("_file_seq_i64") < pl.col("_first_t2_init_seq")))
        .max()
        .alias("_last_t1_init_before_t2"),
        pl.col("_file_seq_i64")
        .filter(pl.col("_is_boot") & (pl.col("_file_seq_i64") <= pl.col("_first_t2_init_seq")))
        .max()
        .alias("_boot_le_t2"),
        pl.col("_file_seq_i64")
        .filter(pl.col("_is_boot") & (pl.col("_file_seq_i64") > pl.col("t1_max_file_seq")))
        .min()
        .alias("_first_boot_after_t1"),
        pl.col("_file_seq_i64")
        .filter(pl.col("_is_boot") & (pl.col("_file_seq_i64") <= pl.col("_first_post_ts_seq")))
        .max()
        .alias("_boot_le_post_ts"),
    )
    del work_anchors

    # Verifica se há atividade de sessão entre t1_max_file_seq e _first_boot_after_t1
    per_jez_gap = (
        work_thin.select(["_jez_key", "_file_seq_i64", "_is_activity", "t1_max_file_seq"])
        .join(
            per_jez_anchors.select(["_jez_key", "_first_boot_after_t1"]),
            on="_jez_key",
            how="left",
        )
        .group_by("_jez_key")
        .agg(
            pl.col("_is_activity")
            .filter(
                (pl.col("_file_seq_i64") > pl.col("t1_max_file_seq"))
                & (pl.col("_file_seq_i64") < pl.col("_first_boot_after_t1"))
            )
            .any()
            .fill_null(False)
            .alias("_activity_before_boot_after_t1"),
        )
    )
    del work_thin

    resolved_jez = per_jez.join(per_jez_anchors, on="_jez_key", how="left").join(
        per_jez_gap, on="_jez_key", how="left"
    )
    del per_jez, per_jez_anchors, per_jez_gap

    last_t1_anchor = pl.max_horizontal(
        pl.lit(0, dtype=pl.Int64),
        pl.when(pl.col("_has_t1_overlap") | pl.col("_already_past_t1"))
        .then(pl.col("_t1_max_seq"))
        .otherwise(pl.lit(0, dtype=pl.Int64)),
        pl.col("_last_t1_close_before_t2").fill_null(0),
        pl.col("_last_t1_init_before_t2").fill_null(0),
    )

    snapped_lifecycle_seq = (
        pl.when(pl.col("_boot_le_t2").is_not_null() & (pl.col("_boot_le_t2") > last_t1_anchor))
        .then(pl.col("_boot_le_t2"))
        .otherwise(pl.col("_first_t2_init_seq"))
    )

    seq_overlap_keep = (
        pl.when(
            pl.col("_first_boot_after_t1").is_not_null()
            & ~pl.col("_activity_before_boot_after_t1").fill_null(False)
        )
        .then(pl.col("_first_boot_after_t1"))
        .otherwise(pl.col("_t1_max_seq") + 1)
    )

    snapped_date_seq = (
        pl.when(
            pl.col("_boot_le_post_ts").is_not_null()
            & (pl.col("_boot_le_post_ts") > pl.col("_last_pre_ts_seq").fill_null(0))
        )
        .then(pl.col("_boot_le_post_ts"))
        .otherwise(pl.col("_first_post_ts_seq"))
    )

    resolved_jez = resolved_jez.with_columns(
        pl.when(pl.col("_first_t2_init_seq").is_not_null())
        .then(
            pl.when(snapped_lifecycle_seq <= pl.col("_min_seq"))
            .then(pl.col("_min_seq"))
            .otherwise(snapped_lifecycle_seq)
        )
        .when(pl.col("_has_t1_overlap"))
        .then(seq_overlap_keep)
        .when(pl.col("_already_past_t1"))
        .then(pl.col("_min_seq"))
        .when(pl.col("_has_pre_cutoff") & pl.col("_has_post_cutoff"))
        .then(snapped_date_seq)
        .otherwise(pl.col("_min_seq"))
        .cast(pl.Int64)
        .alias("_keep_from_seq"),
        pl.when(pl.col("_first_t2_init_seq").is_not_null())
        .then(
            pl.when(snapped_lifecycle_seq <= pl.col("_min_seq"))
            .then(pl.lit("fresh_urna_no_carryover"))
            .otherwise(pl.lit("lifecycle_marker"))
        )
        .when(pl.col("_has_t1_overlap"))
        .then(pl.lit("turn1_urna_seq_overlap"))
        .when(pl.col("_already_past_t1"))
        .then(pl.lit("fresh_urna_no_carryover"))
        .when(pl.col("_has_pre_cutoff") & pl.col("_has_post_cutoff"))
        .then(pl.lit("date_fallback"))
        .when(pl.col("_any_t1_init") | pl.col("_has_pre_cutoff"))
        .then(pl.lit("unresolved"))
        .when(
            pl.col("_has_post_cutoff")
            | (pl.lit(has_history_source) & pl.col("_t1_max_seq").is_null())
        )
        .then(pl.lit("fresh_urna_no_carryover"))
        .otherwise(pl.lit("unresolved"))
        .alias("_resolution_method"),
    )

    method_counts = _empty_resolution_counts()
    for row in (
        resolved_jez.group_by("_resolution_method").agg(pl.len().alias("cnt")).iter_rows(named=True)
    ):
        method_name = str(row["_resolution_method"])
        if method_name in method_counts:
            method_counts[method_name] = int(row["cnt"])

    jez_total = resolved_jez.height

    keep_mask_df = (
        pl.DataFrame({"_jez_key": jez_key_series, "_file_seq_i64": file_seq_series})
        .join(
            resolved_jez.select(["_jez_key", "_keep_from_seq"]),
            on="_jez_key",
            how="left",
            maintain_order="left",
        )
        .select((pl.col("_file_seq_i64") >= pl.col("_keep_from_seq")).alias("_keep"))
    )
    keep_mask = keep_mask_df.get_column("_keep")
    del keep_mask_df, jez_key_series, file_seq_series, resolved_jez

    if "is_clock_drift" not in df.columns:
        result_df = df.with_columns(drift_series.alias("is_clock_drift")).filter(keep_mask)
    else:
        result_df = df.filter(keep_mask)
    del drift_series, keep_mask

    silver_kept = result_df.height
    drift_kept = int(result_df.select(pl.col("is_clock_drift").sum()).item() or 0)

    return result_df, CarryoverFilterStats(
        raw_rows=raw_rows,
        carryover_rows_dropped=raw_rows - silver_kept,
        silver_rows_kept=silver_kept,
        clock_drift_rows_retained=drift_kept,
        jez_files_total=jez_total,
        jez_resolution_counts=method_counts,
    )


def _col_or_default(
    df: pl.DataFrame,
    name: str,
    default_val: object,
    dtype: pl.DataType,
    aliases: tuple[str, ...] = (),
) -> pl.Expr:
    cols = set(df.columns)
    for src_name in (name, *aliases):
        if src_name in cols:
            return pl.col(src_name).cast(dtype)
    return pl.lit(default_val, dtype=dtype).alias(name)


def transform_bronze_shard_to_silver(
    bronze_df: pl.DataFrame,
    *,
    election_year: int = 2026,
    turn: int,
    uf: str,
    catalog: EventCatalog | None = None,
    turn1_history: pl.DataFrame | None = None,
    inter_turn_cutoff: dt_type | None = None,
    parser_version: str = SILVER_PARSER_VERSION,
) -> tuple[pl.DataFrame, CarryoverFilterStats]:
    """Converte um shard Bronze de fact_urna_logs para o schema Silver fact_urna_event."""
    active_catalog = catalog if catalog is not None else CATALOG

    stripped_df, stats = strip_turn1_carryover(
        bronze_df,
        election_year=election_year,
        turn=turn,
        catalog=active_catalog,
        turn1_history=turn1_history,
        inter_turn_cutoff=inter_turn_cutoff,
    )

    if stripped_df.is_empty():
        return pl.DataFrame(schema=SILVER_FACT_URNA_EVENT_SCHEMA), stats

    ts_dtype = stripped_df.schema.get("raw_timestamp")
    parsed_ts_expr = parse_bronze_timestamps_expr("raw_timestamp", dtype=ts_dtype)

    has_app = "application" in stripped_df.columns
    has_msg = "message_text" in stripped_df.columns
    app_norm_expr = (
        pl.col("application").cast(pl.String).fill_null("VOTA").replace("", "VOTA")
        if has_app
        else pl.lit("VOTA", dtype=pl.String)
    )
    msg_norm_expr = (
        pl.col("message_text").cast(pl.String).fill_null("")
        if has_msg
        else pl.lit("", dtype=pl.String)
    )

    # Projeção mínima para regressão temporal (evita ordenar todas as colunas em RAM)
    ts_reg_df = (
        stripped_df.select(
            [
                _jez_key_expr(stripped_df.columns).alias("_jez_key"),
                pl.col("urna_serial").cast(pl.String).fill_null("UNKNOWN").alias("_serial_norm"),
                pl.col("file_seq").cast(pl.Int64).alias("_file_seq_i64"),
                parsed_ts_expr.alias("_parsed_ts"),
            ]
        )
        .with_row_index("_out_idx")
        .sort(["_jez_key", "_file_seq_i64"])
    )

    prev_valid_ts = pl.col("_parsed_ts").forward_fill().shift(1).over(["_jez_key", "_serial_norm"])
    ts_reg_series = (
        ts_reg_df.select(
            [
                pl.col("_out_idx"),
                (
                    pl.col("_parsed_ts").is_not_null()
                    & prev_valid_ts.is_not_null()
                    & (pl.col("_parsed_ts") < prev_valid_ts)
                )
                .fill_null(False)
                .alias("_has_ts_regression"),
            ]
        )
        .sort("_out_idx")
        .get_column("_has_ts_regression")
    )
    del ts_reg_df

    # Classificação deduplicada por (application, message_text) único no shard
    pair_df = stripped_df.select(
        [
            app_norm_expr.alias("_app_norm"),
            msg_norm_expr.alias("_msg_norm"),
        ]
    )
    unique_pairs = pair_df.unique()
    apps_u: list[str] = []
    msgs_u: list[str] = []
    codes_u: list[str] = []
    bio_u: list[int | None] = []
    statuses_u: list[str] = []

    fallback_codes = {active_catalog.fallback_event_code, active_catalog.malformed_event_code}
    for app_val, msg_val in unique_pairs.iter_rows():
        res = active_catalog.classify(msg_val, app_val)
        apps_u.append(app_val)
        msgs_u.append(msg_val)
        codes_u.append(res.event_code)
        bio_u.append(res.biometric_attempt_num)
        statuses_u.append("UNKNOWN" if res.event_code in fallback_codes else "CLASSIFIED")
    del unique_pairs

    lookup_df = pl.DataFrame(
        {
            "_app_norm": pl.Series(apps_u, dtype=pl.String),
            "_msg_norm": pl.Series(msgs_u, dtype=pl.String),
            "event_code": pl.Series(codes_u, dtype=pl.String),
            "biometric_attempt_num": pl.Series(bio_u, dtype=pl.Int8),
            "classification_status": pl.Series(statuses_u, dtype=pl.String),
        }
    )

    classified_cols = pair_df.join(
        lookup_df,
        on=["_app_norm", "_msg_norm"],
        how="left",
        maintain_order="left",
    ).select(["_app_norm", "event_code", "biometric_attempt_num", "classification_status"])
    del pair_df, lookup_df

    silver_df = stripped_df.select(
        [
            _col_or_default(stripped_df, "election_year", election_year, pl.Int16).alias(
                "election_year"
            ),
            _col_or_default(stripped_df, "turn", turn, pl.Int8).alias("turn"),
            _col_or_default(stripped_df, "uf", uf, pl.String).alias("uf"),
            _col_or_default(
                stripped_df, "municipio_code", None, pl.Int32, aliases=("cd_municipio",)
            ).alias("municipio_code"),
            _col_or_default(stripped_df, "zone_num", None, pl.Int16, aliases=("nr_zona",)).alias(
                "zone_num"
            ),
            _col_or_default(
                stripped_df, "section_num", None, pl.Int32, aliases=("nr_secao",)
            ).alias("section_num"),
            _col_or_default(stripped_df, "urna_serial", None, pl.String).alias("urna_serial"),
            pl.col("file_seq").cast(pl.Int64).alias("file_seq"),
            parsed_ts_expr.alias("raw_timestamp"),
            pl.col("is_clock_drift").cast(pl.Boolean).alias("is_clock_drift"),
            pl.lit(False, dtype=pl.Boolean).alias("has_timestamp_regression"),
            pl.lit("VOTA", dtype=pl.String).alias("application"),
            _col_or_default(stripped_df, "log_level", None, pl.String).alias("log_level"),
        ]
    ).with_columns(
        ts_reg_series.cast(pl.Boolean).alias("has_timestamp_regression"),
        classified_cols.get_column("_app_norm").alias("application"),
        classified_cols.get_column("event_code").cast(pl.String).alias("event_code"),
        classified_cols.get_column("biometric_attempt_num")
        .cast(pl.Int8)
        .alias("biometric_attempt_num"),
        classified_cols.get_column("classification_status")
        .cast(pl.String)
        .alias("classification_status"),
        pl.lit(parser_version, dtype=pl.String).alias("parser_version"),
    ).select(list(SILVER_FACT_URNA_EVENT_SCHEMA.names()))
    del stripped_df, ts_reg_series, classified_cols

    return silver_df, stats

# Quick smoke test
test_result = classify_row("Voto do eleitor computado", "VOTA", CATALOG)
assert test_result["event_code"] == "VOTER_VOTE_COMPUTED", f"Smoke test falhou: {test_result}"
print(f"classify_row + carryover helpers OK: {test_result}")

In [ ]:
# ---------------------------------------------------------------------------
# Main loop: Bronze -> Turn 2 Carryover Strip -> Classify -> Silver
# ---------------------------------------------------------------------------
import time

bronze_total = 0
carryover_dropped_total = 0
silver_total = 0
clock_drift_retained_total = 0
skipped_partitions: list[str] = []
partitions_audit: list[dict[str, Any]] = []
jez_resolution_totals: dict[str, int] = {m: 0 for m in RESOLUTION_METHODS}

t0 = time.time()

turn1_history_df: pl.DataFrame | None = None
processed_turn1_ufs: set[str] = set()


def _process_partition(turn: int, uf: str) -> None:
    global bronze_total, carryover_dropped_total, silver_total, clock_drift_retained_total, turn1_history_df

    part_dir = BRONZE_BASE / f"election_year={ELECTION_YEAR}" / f"turn={turn}" / f"uf={uf}"
    parquet_files = sorted(part_dir.glob("*.parquet")) if part_dir.exists() else []

    if not parquet_files:
        skipped_partitions.append(f"turn={turn}/uf={uf}")
        return

    local_part_stage = (
        Path(STAGING_PATH)
        / "silver_fact_urna_event"
        / f"election_year={ELECTION_YEAR}"
        / f"turn={turn}"
        / f"uf={uf}"
    )
    if local_part_stage.exists():
        shutil.rmtree(local_part_stage, ignore_errors=True)
    local_part_stage.mkdir(parents=True, exist_ok=True)

    part_bronze_rows = 0
    part_dropped_rows = 0
    part_silver_rows = 0
    part_drift_rows = 0
    part_jez_total = 0
    part_res_counts = {m: 0 for m in RESOLUTION_METHODS}
    staged_files: list[Path] = []

    try:
        for pq_file in parquet_files:
            bronze_df = pl.read_parquet(pq_file)

            if turn == 1:
                t1_chunk = extract_turn1_shard_history(bronze_df)
                if not t1_chunk.is_empty():
                    turn1_history_df = merge_turn1_history(turn1_history_df, t1_chunk)
                del t1_chunk

            silver_df, shard_stats = transform_bronze_shard_to_silver(
                bronze_df,
                election_year=ELECTION_YEAR,
                turn=turn,
                uf=uf,
                catalog=CATALOG,
                turn1_history=turn1_history_df if turn == 2 else None,
                parser_version=PARSER_VERSION,
            )

            part_bronze_rows += shard_stats.raw_rows
            part_dropped_rows += shard_stats.carryover_rows_dropped
            part_silver_rows += shard_stats.silver_rows_kept
            part_drift_rows += shard_stats.clock_drift_rows_retained
            part_jez_total += shard_stats.jez_files_total
            for m_key, m_cnt in shard_stats.jez_resolution_counts.items():
                part_res_counts[m_key] = part_res_counts.get(m_key, 0) + m_cnt

            if not silver_df.is_empty():
                local_out = local_part_stage / f"part-{len(staged_files)}.parquet"
                silver_df.write_parquet(local_out, compression="zstd", compression_level=5)
                staged_files.append(local_out)

            del bronze_df, silver_df

        # Drive FUSE safety: limpa partição no destino antes de copiar shards finalizados
        out_dir = SILVER_OUT / f"election_year={ELECTION_YEAR}" / f"turn={turn}" / f"uf={uf}"
        if out_dir.exists():
            shutil.rmtree(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        for local_out in staged_files:
            dest_out = out_dir / local_out.name
            shutil.copyfile(local_out, dest_out)
            if dest_out.stat().st_size != local_out.stat().st_size:
                raise OSError(f"Tamanho divergente após cópia para {dest_out}")
    finally:
        shutil.rmtree(local_part_stage, ignore_errors=True)

    bronze_total += part_bronze_rows
    carryover_dropped_total += part_dropped_rows
    silver_total += part_silver_rows
    clock_drift_retained_total += part_drift_rows
    for m_key, m_cnt in part_res_counts.items():
        jez_resolution_totals[m_key] = jez_resolution_totals.get(m_key, 0) + m_cnt

    partitions_audit.append({
        "uf": uf,
        "turn": int(turn),
        "raw_rows": part_bronze_rows,
        "carryover_rows_dropped": part_dropped_rows,
        "silver_rows_kept": part_silver_rows,
        "clock_drift_rows_retained": part_drift_rows,
        "jez_files_total": part_jez_total,
        "jez_resolution_counts": part_res_counts,
    })

    print(
        f"  turn={turn} uf={uf}: raw={part_bronze_rows:,} | dropped={part_dropped_rows:,} | "
        f"kept={part_silver_rows:,} | drift={part_drift_rows:,} ({len(staged_files)} part(s))"
    )


if 1 in TURNS:
    for uf in TARGET_UFS:
        _process_partition(1, uf)
        processed_turn1_ufs.add(uf)

if 2 in TURNS:
    # Garante histórico completo do 1º turno mesmo em execuções parciais (TSE_TARGET_TURNS=2 ou UFs filtradas)
    t1_base = BRONZE_BASE / f"election_year={ELECTION_YEAR}" / "turn=1"
    if t1_base.exists():
        extra_t1_files = [
            pq_f
            for uf_dir in sorted(t1_base.glob("uf=*"))
            if uf_dir.name.removeprefix("uf=") not in processed_turn1_ufs
            for pq_f in sorted(uf_dir.glob("*.parquet"))
        ]
        if extra_t1_files:
            extra_hist = scan_turn1_urna_history(extra_t1_files)
            if not extra_hist.is_empty():
                turn1_history_df = merge_turn1_history(turn1_history_df, extra_hist)
            del extra_hist

    for uf in TARGET_UFS:
        _process_partition(2, uf)

for turn in TURNS:
    if turn not in (1, 2):
        for uf in TARGET_UFS:
            _process_partition(turn, uf)

elapsed = time.time() - t0
print(f"\nLoop concluído em {elapsed:.1f}s")
print(f"Partições ignoradas (sem dados Bronze): {len(skipped_partitions)}")
print(
    f"Bronze raw: {bronze_total:,} | Carryover dropped: {carryover_dropped_total:,} | "
    f"Silver kept: {silver_total:,} | Clock-drift retained: {clock_drift_retained_total:,}"
)

In [ ]:
# ---------------------------------------------------------------------------
# Reconciliação: Conservação de linhas + Auditoria de Carryover por (uf, turn)
# ---------------------------------------------------------------------------
if bronze_total == 0:
    raise RuntimeError("Nenhuma linha Bronze foi lida: a Silver ficaria vazia. Verifique o dataset de entrada.")

for part in partitions_audit:
    if part["turn"] == 1 and part["carryover_rows_dropped"] != 0:
        raise AssertionError(
            f"RECONCILIATION FAILED: turn=1 uf={part['uf']} teve {part['carryover_rows_dropped']} linhas removidas!"
        )
    if part["raw_rows"] != part["carryover_rows_dropped"] + part["silver_rows_kept"]:
        raise AssertionError(
            f"RECONCILIATION FAILED em turn={part['turn']} uf={part['uf']}: "
            f"{part['raw_rows']} != {part['carryover_rows_dropped']} + {part['silver_rows_kept']}"
        )

if bronze_total != carryover_dropped_total + silver_total:
    raise AssertionError(
        f"RECONCILIATION FAILED: bronze_total={bronze_total:,} != "
        f"carryover_dropped_total={carryover_dropped_total:,} + silver_total={silver_total:,}"
    )

print(
    f"Reconciliação OK: {bronze_total:,} Bronze == "
    f"{carryover_dropped_total:,} carryover dropped + {silver_total:,} Silver kept."
)
print("\n--- Auditoria por (turn, uf) ---")
print(
    f"{'TURN':>4} | {'UF':<2} | {'RAW_ROWS':>12} | {'DROPPED':>12} | {'KEPT':>12} | "
    f"{'DRIFT':>8} | {'LIFECYCLE':>9} | {'T1_SEQ':>7} | {'DATE_FB':>7} | {'FRESH':>7} | {'UNRES':>6}"
)
print("-" * 112)
for p in partitions_audit:
    rc = p["jez_resolution_counts"]
    print(
        f"{p['turn']:>4} | {p['uf']:<2} | {p['raw_rows']:>12,} | {p['carryover_rows_dropped']:>12,} | "
        f"{p['silver_rows_kept']:>12,} | {p['clock_drift_rows_retained']:>8,} | "
        f"{rc.get('lifecycle_marker', 0):>9,} | {rc.get('turn1_urna_seq_overlap', 0):>7,} | "
        f"{rc.get('date_fallback', 0):>7,} | {rc.get('fresh_urna_no_carryover', 0):>7,} | "
        f"{rc.get('unresolved', 0):>6,}"
    )

In [ ]:
# ---------------------------------------------------------------------------
# Coverage report + manifest (staged safely before Kaggle upload)
# ---------------------------------------------------------------------------
silver_all_files = sorted(SILVER_OUT.rglob("*.parquet"))
print(f"Arquivos Silver gerados: {len(silver_all_files)}")

if silver_all_files:
    silver_lf = pl.scan_parquet([str(p) for p in silver_all_files])

    print("\n--- event_code value counts ---")
    event_code_counts = (
        silver_lf
        .group_by("event_code")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )
    print(event_code_counts)

    print("\n--- classification_status breakdown ---")
    status_counts = (
        silver_lf
        .group_by("classification_status")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )
    print(status_counts)

    total_silver = int(silver_lf.select(pl.len()).collect().item())
    unknown_count = int(
        silver_lf
        .filter(pl.col("event_code") == "UNKNOWN_EVENT")
        .select(pl.len())
        .collect()
        .item()
    )
    unknown_pct = 100.0 * unknown_count / total_silver if total_silver > 0 else 0.0
    print(f"\nUNKNOWN_EVENT: {unknown_count:,} / {total_silver:,} = {unknown_pct:.2f}%")
    print(f"is_clock_drift=True retidas: {clock_drift_retained_total:,}")

    manifest = {
        "notebook": "05_silver_fact_urna_event",
        "parser_version": PARSER_VERSION,
        "election_year": ELECTION_YEAR,
        "turns": TURNS,
        "bronze_total_rows": bronze_total,
        "carryover_dropped_total": carryover_dropped_total,
        "silver_total_rows": silver_total,
        "clock_drift_rows_retained": clock_drift_retained_total,
        "jez_resolution_totals": jez_resolution_totals,
        "partitions_audit": partitions_audit,
        "silver_parquet_files": len(silver_all_files),
        "unknown_event_count": unknown_count,
        "unknown_event_pct": round(unknown_pct, 4),
        "classification_status_counts": {
            row["classification_status"]: row["count"]
            for row in status_counts.to_dicts()
        },
        "timestamp_utc": __import__("datetime").datetime.now(__import__("datetime").UTC).isoformat(),
    }
    print("\n--- Manifest ---")
    print(json.dumps(manifest, indent=2, ensure_ascii=False))

    # Persist manifest safely via staging (Drive FUSE rule)
    manifest_path = SILVER_OUT / "manifest.json"
    manifest_tmp = Path(STAGING_PATH) / "nb05_manifest.json.tmp"
    manifest_tmp.parent.mkdir(parents=True, exist_ok=True)
    manifest_tmp.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    if manifest_path.exists():
        manifest_path.unlink()
    shutil.copyfile(manifest_tmp, manifest_path)
    manifest_tmp.unlink(missing_ok=True)
    print(f"\nManifesto salvo em: {manifest_path}")
else:
    print("Nenhum arquivo Silver encontrado — verifique se o Bronze contém dados para os turnos/UFs configurados.")

# Validação de higiene pré-upload e publicação de nova versão via KaggleHub SDK
PHASE_ID = "05_silver_fact_urna_event"
PHASE_SUMMARY = "Camada Silver: deduplicação de carryover T1->T2 e classificação semântica em fact_urna_event (Parquet ZSTD)"
forbidden_extensions = {".zip", ".jez", ".tmp", ".part", ".log"}
offending_files = [
    str(p.relative_to(DATASET_PATH))
    for p in Path(DATASET_PATH).rglob("*")
    if p.is_file() and any(sfx.lower() in forbidden_extensions for sfx in p.suffixes)
]
if offending_files:
    raise ValueError(
        f"Upload abortado! Arquivos brutos/temporários detectados em DATASET_PATH: {offending_files[:10]}"
    )

if is_directory_empty(Path(DATASET_PATH)):
    raise RuntimeError(
        f"Upload abortado! DATASET_PATH ({DATASET_PATH}) está vazio após o processamento."
    )

if globals().get("__name__") == "__main__" and not os.getenv("PYTEST_CURRENT_TEST") and not os.getenv("TSE_SKIP_KAGGLE_RELEASE"):
    import kagglehub

    print(f"Publicando nova versão do dataset '{DATASET_HANDLE}' a partir de {DATASET_PATH}...")
    kagglehub.dataset_upload(
        handle=DATASET_HANDLE,
        local_dataset_dir=str(DATASET_PATH),
        version_notes=f"[{PHASE_ID}] {PHASE_SUMMARY}",
        ignore_patterns=[
            "*.tmp",
            "*.part",
            "*.zip",
            "*.jez",
            "*.log",
            "__pycache__/",
            ".ipynb_checkpoints/",
            ".DS_Store",
        ],
    )
    print(f"[SUCESSO] Nova versão de '{DATASET_HANDLE}' publicada no Kaggle!")

## Limitações e Notas

- **Deduplicação de carryover no 2º turno**: Quando uma urna eletrônica é reutilizada no 2º turno, o `logd.dat` preserva o histórico do 1º turno como prefixo. A Silver remove esse prefixo exclusivamente para `turn == 2` utilizando: (1) marcadores de ciclo de vida do 2º turno (`turn_boundary_patterns`) em `file_seq`, (2) histórico de `urna_serial` do 1º turno (`t1_max_file_seq`) e (3) fallback de data apenas sobre timestamps sem *clock drift*. A camada Bronze permanece 1:1 intacta.

- **Clock drift de BIOS (`is_clock_drift`)**: Urnas com reset de relógio (ex.: `01/01/1970 00:00:53`) recebem `is_clock_drift = True` e jamais são descartadas pelo filtro de carryover com base na data isolada.

- **Cobertura do catálogo**: O `event_catalog.yml` cobre 16 famílias de eventos oficialmente documentadas pelo TSE. Mensagens que não correspondam a nenhum padrão recebem `event_code = UNKNOWN_EVENT` e `classification_status = UNKNOWN`.

- **Extração de PII**: Nenhuma linha retém texto livre nem dados pessoais. O único valor numérico extraído é `biometric_attempt_num` (1–127).

- **Idempotência**: A execução é idempotente — Parquet Silver existentes são substituídos com segurança via staging local (`/tmp/tse-eleicoes-staging`).